# Task 4: Open-Set Recognition — GPU Colab (Drive-first)

For **any Google account that still has GPU quota** (code Drive may be out of quota).

**Safety model:** every artifact is written **directly into this account’s Google Drive**.
`/content` is only a scratch clone of the git repo; results live on Drive via a symlink,
so a runtime disconnect does not delete checkpoints/tables/figures.

### What gets saved on Drive
`MyDrive/ATML-PA1-task4-backup/`
- `results/checkpoints/*.pt` — Vanilla / GCSC / PROSER
- `results/curves/*_history.json`
- `results/tables/*.json` — score + model comparison + failures
- `results/figures/score_distributions.png`
- `results/cache/**` — logits/features
- `results/splits/cifar10_seed6304.json`
- `task4_results_bundle.zip` — full zip (refreshed after each stage)
- `task4_pipeline.log` — training log

### After DONE
Copy `task4_results_bundle.zip` → your code account → unzip into `task4/results/`.

**Hard rules:** CIFAR-10 val Acc for checkpoints; CIFAR-10 val only for thresholds; no CIFAR-100 in train.

In [ ]:
import torch

print("cuda:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("No GPU — Runtime → Change runtime type → GPU, then re-run.")
print("device:", torch.cuda.get_device_name(0))

## 1) Mount **this** account’s Google Drive

Sign into Colab with the GPU account. Drive = that same account.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/ATML-PA1-task4-backup")
DRIVE_RESULTS = DRIVE_ROOT / "results"
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
print("Drive root:", DRIVE_ROOT)
print("Existing checkpoints:")
for p in sorted((DRIVE_RESULTS / "checkpoints").glob("*.pt")) if (DRIVE_RESULTS / "checkpoints").exists() else []:
    print(f"  {p.name}  {p.stat().st_size} bytes")

## 2) Clone GitHub code into `/content` (ephemeral)

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/ttqureshi/ATML-PA1.git"
REPO_DIR = Path("/content/ATML-PA1")

if (REPO_DIR / ".git").is_dir():
    print("Repo exists — pulling latest main...")
    subprocess.check_call(["git", "-C", str(REPO_DIR), "fetch", "origin"])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "checkout", "main"])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"])
else:
    print("Cloning", REPO_URL)
    subprocess.check_call(["git", "clone", "--branch", "main", REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
print("cwd:", Path.cwd())
!git log -1 --oneline

In [ ]:
%pip install -q -r requirements.txt

## 3) Point `task4/results` → Drive (symlink)

All training/eval writes go straight to Drive. Disconnect-safe.

In [ ]:
from pathlib import Path
import shutil

REPO_DIR = Path("/content/ATML-PA1")
LOCAL_RESULTS = REPO_DIR / "task4" / "results"
DRIVE_ROOT = Path("/content/drive/MyDrive/ATML-PA1-task4-backup")
DRIVE_RESULTS = DRIVE_ROOT / "results"
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

# If local results already exists as a real dir, merge into Drive then replace with symlink
if LOCAL_RESULTS.exists() or LOCAL_RESULTS.is_symlink():
    if LOCAL_RESULTS.is_symlink():
        LOCAL_RESULTS.unlink()
    elif LOCAL_RESULTS.is_dir():
        print("Merging existing local results into Drive...")
        shutil.copytree(LOCAL_RESULTS, DRIVE_RESULTS, dirs_exist_ok=True)
        shutil.rmtree(LOCAL_RESULTS)

LOCAL_RESULTS.parent.mkdir(parents=True, exist_ok=True)
LOCAL_RESULTS.symlink_to(DRIVE_RESULTS)

assert LOCAL_RESULTS.resolve() == DRIVE_RESULTS.resolve()
print("OK: task4/results →", LOCAL_RESULTS.resolve())

def refresh_zip_and_log(tag: str = ""):
    """Rebuild the downloadable zip + copy pipeline log onto Drive."""
    zip_base = DRIVE_ROOT / "task4_results_bundle"
    zip_path = zip_base.with_suffix(".zip")
    if zip_path.exists():
        zip_path.unlink()
    shutil.make_archive(str(zip_base), "zip", root_dir=DRIVE_RESULTS)
    log_src = Path("/content/task4_pipeline.log")
    if log_src.exists():
        shutil.copy2(log_src, DRIVE_ROOT / "task4_pipeline.log")
    print(f"[{tag}] zip → {zip_path} ({zip_path.stat().st_size} bytes)")

refresh_zip_and_log("init")

## 4) Start full pipeline (background)

Resume-aware: skips any train stage whose `*_best.pt` already exists on Drive.
Rebuilds the Drive zip after every stage.

In [ ]:
import os, subprocess
from pathlib import Path

os.chdir("/content/ATML-PA1")
log = Path("/content/task4_pipeline.log")
done = Path("/content/task4_pipeline.done")
if done.exists():
    done.unlink()
log.write_text("starting Drive-first pipeline\n")

script = r'''
set -e
cd /content/ATML-PA1
CKPT=task4/results/checkpoints
mkdir -p "$CKPT"
DRIVE_ROOT=/content/drive/MyDrive/ATML-PA1-task4-backup

refresh() {
  TAG="$1"
  python - <<PY
from pathlib import Path
import shutil
DRIVE_ROOT = Path("$DRIVE_ROOT")
RESULTS = Path("task4/results").resolve()
zip_base = DRIVE_ROOT / "task4_results_bundle"
zip_path = zip_base.with_suffix(".zip")
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_base), "zip", root_dir=RESULTS)
log_src = Path("/content/task4_pipeline.log")
if log_src.exists():
    shutil.copy2(log_src, DRIVE_ROOT / "task4_pipeline.log")
print("[", "$TAG", "] zip", zip_path, zip_path.stat().st_size)
# inventory
for sub in ["checkpoints", "tables", "curves", "figures", "splits", "cache"]:
    p = RESULTS / sub
    n = sum(1 for _ in p.rglob("*") if _.is_file()) if p.exists() else 0
    print(f"  {sub}: {n} files")
PY
}

python -m task4.scripts.run_task4 --stages splits
refresh splits

if [ ! -f "$CKPT/vanilla_best.pt" ]; then
  python -m task4.scripts.run_task4 --stages train_vanilla
  refresh vanilla
else
  echo "SKIP train_vanilla (Drive checkpoint exists)"
fi

if [ ! -f "$CKPT/gcsc_best.pt" ]; then
  python -m task4.scripts.run_task4 --stages train_gcsc
  refresh gcsc
else
  echo "SKIP train_gcsc (Drive checkpoint exists)"
fi

if [ ! -f "$CKPT/proser_best.pt" ]; then
  python -m task4.scripts.run_task4 --stages train_proser
  refresh proser
else
  echo "SKIP train_proser (Drive checkpoint exists)"
fi

python -m task4.scripts.run_task4 --stages extract_all
refresh extract

python -m task4.scripts.run_task4 --stages eval
refresh eval

touch /content/task4_pipeline.done
cp /content/task4_pipeline.done "$DRIVE_ROOT/" || true
echo DONE
'''

proc = subprocess.Popen(
    ["bash", "-lc", script],
    stdout=open(log, "a"),
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print("PID", proc.pid)
print("Live log: /content/task4_pipeline.log")
print("Durable Drive folder: /content/drive/MyDrive/ATML-PA1-task4-backup/")
print("Keep the Colab tab open. Even if the runtime dies mid-stage, finished *_best.pt files stay on Drive.")

## 5) Poll progress (re-run this cell anytime)

In [ ]:
from pathlib import Path
import subprocess

DRIVE_ROOT = Path("/content/drive/MyDrive/ATML-PA1-task4-backup")
log = Path("/content/task4_pipeline.log")
done = Path("/content/task4_pipeline.done")
print("local done:", done.exists(), "| Drive done:", (DRIVE_ROOT / "task4_pipeline.done").exists())

r = subprocess.run(["bash", "-lc", "pgrep -af 'python3 -m task4' || true"], capture_output=True, text=True)
print("procs:\n", r.stdout)

if log.exists():
    lines = log.read_text(errors="ignore").splitlines()
    keep = [
        ln for ln in lines
        if ("epoch" in ln.lower() or ">>" in ln or "best" in ln or "SKIP" in ln
            or "zip" in ln.lower() or "DONE" in ln or "Error" in ln or "Traceback" in ln
            or "Wrote" in ln or "Evaluating" in ln or "files" in ln)
        and "it/s" not in ln
    ]
    print("--- log tail ---")
    print("\n".join(keep[-50:]))

print("\n=== Drive inventory ===")
for sub in ["checkpoints", "tables", "curves", "figures", "splits", "cache"]:
    p = DRIVE_ROOT / "results" / sub
    if not p.exists():
        print(f"{sub}: (missing)")
        continue
    files = [x for x in p.rglob("*") if x.is_file()]
    print(f"{sub}: {len(files)} files")
    for x in sorted(files)[:12]:
        print(f"  {x.relative_to(DRIVE_ROOT)}  ({x.stat().st_size} B)")

zp = DRIVE_ROOT / "task4_results_bundle.zip"
print("\nzip:", zp.exists(), zp.stat().st_size if zp.exists() else "")

## 6) After DONE — move to code account

On **this GPU account’s Drive**:
`MyDrive/ATML-PA1-task4-backup/task4_results_bundle.zip`

1. Download / share that zip to your code (account A) Drive.
2. Unzip into `ATML-PA1/task4/results/` on account A.
3. Tell the agent to verify tables/figures and interpret RQs.

If the runtime dies mid-run: reconnect GPU → remount Drive → re-run cells 1–4.
Finished checkpoints are skipped automatically.